# 에너지 수요 예측 데이터 전처리
**대회**: Enefit – Predict Energy Behavior of Prosumers 

---


## 목차
1. [경로 설정 및 로드](#load)
2. [데이터 전처리](#preprocessing)
   - 2.1 datetime 변환
   - 2.2 시간 파생 변수 생성
   - 2.3 필요한 컬럼 선택
3. [데이터 병합](#merge)
   - 3.1 Client 병합
   - 3.2 Weather 병합
   - 3.3 Electricity 병합
   - 3.4 Gas 병합
4. [Feature Engineering](#features)
   - 4.1 정렬 및 lag feature
   - 4.2 모델별 Feature 세트 정의
5. [결측치 처리](#missing)
6. [시계열 Split](#split)
7. [CSV 저장](#save)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

<a id="load"></a>
## 1️ 경로 설정 및 로드

프로젝트 경로를 설정하고 모든 필요한 데이터셋을 로드합니다.

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

# 파일명
TRAIN_PATH = DATA_DIR / "train.csv"
CLIENT_PATH = DATA_DIR / "client.csv"
ELECTRICITY_PATH = DATA_DIR / "electricity_prices.csv"
GAS_PATH = DATA_DIR / "gas_prices.csv"
HISTORICAL_WEATHER_PATH = DATA_DIR / "historical_weather.csv"
WEATHER_TO_COUNTY_PATH = DATA_DIR / "weather_station_to_county_mapping.csv"



# 저장 경로
OUTPUT_DIR = DATA_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
train = pd.read_csv(TRAIN_PATH)
client = pd.read_csv(CLIENT_PATH)
electricity = pd.read_csv(ELECTRICITY_PATH)
gas = pd.read_csv(GAS_PATH)
historical_weather = pd.read_csv(HISTORICAL_WEATHER_PATH)
weather_to_county = pd.read_csv(WEATHER_TO_COUNTY_PATH)


print("train:", train.shape)
print("client:", client.shape)
print("electricity:", electricity.shape)
print("gas:", gas.shape)

print("historical_weather:", historical_weather.shape)
print("weather_to_county:", weather_to_county.shape)

train: (2018352, 9)
client: (41919, 7)
electricity: (15286, 4)
gas: (637, 5)
historical_weather: (1710802, 18)
weather_to_county: (112, 4)


<a id="preprocessing"></a>
## 2️ 데이터 전처리

날짜 변환, 파생 변수 생성, 필요한 컬럼을 선택합니다.

In [4]:
# =========================================================
# 2. datetime conversion
# =========================================================
train["datetime"] = pd.to_datetime(train["datetime"])
client["date"] = pd.to_datetime(client["date"])
electricity["forecast_date"] = pd.to_datetime(electricity["forecast_date"])
gas["forecast_date"] = pd.to_datetime(gas["forecast_date"])
historical_weather["datetime"] = pd.to_datetime(historical_weather["datetime"])

# normalize date columns for merges
train["date"] = train["datetime"].dt.normalize()
client["date"] = client["date"].dt.normalize()
gas["date"] = gas["forecast_date"].dt.normalize()

print("\n[datetime converted]")
print(train[["datetime", "date"]].head())

# =========================================================
# 3. time features
# =========================================================
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["dayofyear"] = train["datetime"].dt.dayofyear
train["weekofyear"] = train["datetime"].dt.isocalendar().week.astype("int16")
train["quarter"] = train["datetime"].dt.quarter.astype("int8")
train["is_weekend"] = (train["weekday"] >= 5).astype("int8")
train["is_month_start"] = train["datetime"].dt.is_month_start.astype("int8")
train["is_month_end"] = train["datetime"].dt.is_month_end.astype("int8")
train["is_quarter_start"] = train["datetime"].dt.is_quarter_start.astype("int8")
train["is_quarter_end"] = train["datetime"].dt.is_quarter_end.astype("int8")
train["days_in_month"] = train["datetime"].dt.days_in_month.astype("int8")

print("\n[time features created]")
print(train[[
    "datetime", "hour", "weekday", "month", "day", "dayofyear", "weekofyear",
    "quarter", "is_weekend", "is_month_start", "is_month_end",
    "is_quarter_start", "is_quarter_end", "days_in_month"
]].head())



[datetime converted]
    datetime       date
0 2021-09-01 2021-09-01
1 2021-09-01 2021-09-01
2 2021-09-01 2021-09-01
3 2021-09-01 2021-09-01
4 2021-09-01 2021-09-01

[time features created]
    datetime  hour  weekday  month  day  dayofyear  weekofyear  quarter  \
0 2021-09-01     0        2      9    1        244          35        3   
1 2021-09-01     0        2      9    1        244          35        3   
2 2021-09-01     0        2      9    1        244          35        3   
3 2021-09-01     0        2      9    1        244          35        3   
4 2021-09-01     0        2      9    1        244          35        3   

   is_weekend  is_month_start  is_month_end  is_quarter_start  is_quarter_end  \
0           0               1             0                 0               0   
1           0               1             0                 0               0   
2           0               1             0                 0               0   
3           0               1     

In [5]:
# =========================================================
# 4. keep required columns
# =========================================================
train_cols = [
    "row_id",
    "prediction_unit_id",
    "county",
    "is_business",
    "product_type",
    "is_consumption",
    "target",
    "datetime",
    "date",
    "hour",
    "weekday",
    "month",
    "day",
    "dayofyear",
    "weekofyear",
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_quarter_start",
    "is_quarter_end",
    "days_in_month",
    "data_block_id",
]
train = train[train_cols].copy()

client_cols = [
    "county",
    "is_business",
    "product_type",
    "date",
    "eic_count",
    "installed_capacity",
]
client = client[client_cols].copy()

historical_weather_cols = [
    "datetime",
    "temperature",
    "dewpoint",
    "rain",
    "snowfall",
    "surface_pressure",
    "cloudcover_total",
    "windspeed_10m",
    "shortwave_radiation",
    "direct_solar_radiation",
    "latitude",
    "longitude",
]
historical_weather = historical_weather[historical_weather_cols].copy()

print("\n[required columns kept]")

electricity = electricity[["forecast_date", "euros_per_mwh"]].copy()
gas = gas[["forecast_date", "date", "lowest_price_per_mwh", "highest_price_per_mwh"]].copy()



[required columns kept]


<a id="merge"></a>
## 3️ 데이터 병합

Client, Weather, Electricity, Gas 테이블들을 datetime 기준으로 병합합니다.

In [6]:
# =========================================================
# 5. client merge
# =========================================================
train = train.merge(
    client,
    on=["county", "is_business", "product_type", "date"],
    how="left"
)

print("\n[client merge 완료]")
print("shape:", train.shape)

# =========================================================
# 6. weather merge
#    여러 weather station 평균값을 datetime 기준으로 집계
#    추가: county별 weather를 계산해 XGBoost와 공통 feature로 만듦
# =========================================================
historical_weather['datetime'] = pd.to_datetime(historical_weather['datetime'])
county_mapping_clean = weather_to_county.dropna(subset=['county']).copy()

county_mapping_clean['county'] = county_mapping_clean['county'].astype(int)

hw_county = historical_weather.merge(
    county_mapping_clean[['latitude', 'longitude', 'county']],
    on=['latitude', 'longitude'], how='inner'
)


weather_county = hw_county.groupby(["county", "datetime"], as_index=False).agg(
    temp_county     = ("temperature",            "mean"),
    solar_county    = ("shortwave_radiation",    "mean"),
    direct_solar    = ("direct_solar_radiation", "mean"),
    cloud_county    = ("cloudcover_total",       "mean"),
    wind_county     = ("windspeed_10m",          "mean"),
)

print("\n[county별 weather 집계 완료]")
print("weather_county:", weather_county.shape)

weather_agg = historical_weather.groupby("datetime", as_index=False).agg(
    temperature=("temperature", "mean"),
    dewpoint=("dewpoint", "mean"),
    rain=("rain", "mean"),
    snowfall=("snowfall", "mean"),
    surface_pressure=("surface_pressure", "mean"),
    cloudcover_total=("cloudcover_total", "mean"),
    windspeed_10m=("windspeed_10m", "mean"),
    shortwave_radiation=("shortwave_radiation", "mean"),
    direct_solar_radiation=("direct_solar_radiation", "mean"),
)

train = train.merge(
    weather_agg,
    on="datetime",
    how="left"
)

train = train.merge(
    weather_county,
    on=["county", "datetime"],
    how="left"
)

print("\n[historical_weather merge 완료]")
print("shape:", train.shape)

# =========================================================
# 7. electricity merge
# =========================================================
electricity = electricity.rename(columns={"forecast_date": "datetime"})

train = train.merge(
    electricity,
    on="datetime",
    how="left"
)

print("\n[electricity merge 완료]")
print("shape:", train.shape)

# =========================================================
# 8. gas merge
#    gas는 날짜 단위 기준으로 merge
# =========================================================
train = train.merge(
    gas[["date", "lowest_price_per_mwh", "highest_price_per_mwh"]],
    on="date",
    how="left"
)

print("\n[gas merge 완료]")
print("shape:", train.shape)



[client merge 완료]
shape: (2018352, 25)

[county별 weather 집계 완료]
weather_county: (106925, 7)

[historical_weather merge 완료]
shape: (2018352, 39)

[electricity merge 완료]
shape: (2018352, 40)

[gas merge 완료]
shape: (2018352, 42)


In [7]:
# =========================================================
# 9. sort before lag features
# =========================================================
train = train.sort_values(
    ["prediction_unit_id", "is_consumption", "datetime"]
).reset_index(drop=True)

print("\n[sorted]")
print(train[["prediction_unit_id", "is_consumption", "datetime"]].head())

# =========================================================
# 10. lag features
# =========================================================
group_cols = ["prediction_unit_id", "is_consumption"]

train["lag_1"] = train.groupby(group_cols)["target"].shift(1).astype("float32")
train["lag_24"] = train.groupby(group_cols)["target"].shift(24).astype("float32")

print("\n[lag features created]")
print(train[["target", "lag_1", "lag_24"]].head(30))

# =========================================================
# 11. feature engineering
# =========================================================

# 11-1. lag features: short, mid, and weekly target lags
for lag in [2, 3, 6, 12, 48, 72, 96, 120, 168]:
    train[f"lag_{lag}"] = train.groupby(group_cols)["target"].shift(lag).astype("float32")

# 11-2. rolling statistics: short, mid, and long windows
for window in [3, 6, 12, 24, 48, 168]:
    train[f"rolling_mean_{window}"] = (
        train.groupby(group_cols)["target"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype("float32")
    )
    train[f"rolling_std_{window}"] = (
        train.groupby(group_cols)["target"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
        .astype("float32")
    )

# 11-3. cyclic encodings
train["hour_sin"] = np.sin(2 * np.pi * train["hour"] / 24)
train["hour_cos"] = np.cos(2 * np.pi * train["hour"] / 24)
train["dow_sin"] = np.sin(2 * np.pi * train["weekday"] / 7)
train["dow_cos"] = np.cos(2 * np.pi * train["weekday"] / 7)
train["month_sin"] = np.sin(2 * np.pi * (train["month"] - 1) / 12)
train["month_cos"] = np.cos(2 * np.pi * (train["month"] - 1) / 12)
train["day_sin"] = np.sin(2 * np.pi * (train["day"] - 1) / train["days_in_month"])
train["day_cos"] = np.cos(2 * np.pi * (train["day"] - 1) / train["days_in_month"])
train["dayofyear_sin"] = np.sin(2 * np.pi * (train["dayofyear"] - 1) / 365.25)
train["dayofyear_cos"] = np.cos(2 * np.pi * (train["dayofyear"] - 1) / 365.25)
train["weekofyear_sin"] = np.sin(2 * np.pi * (train["weekofyear"] - 1) / 52)
train["weekofyear_cos"] = np.cos(2 * np.pi * (train["weekofyear"] - 1) / 52)
train["quarter_sin"] = np.sin(2 * np.pi * (train["quarter"] - 1) / 4)
train["quarter_cos"] = np.cos(2 * np.pi * (train["quarter"] - 1) / 4)

print("\n[feature engineering created]")
print(train[[
    "lag_2", "lag_3", "lag_6", "lag_12",
    "lag_48", "lag_72", "lag_96", "lag_120", "lag_168",
    "rolling_mean_3", "rolling_std_3",
    "rolling_mean_24", "rolling_std_24",
    "rolling_mean_48", "rolling_std_48",
    "rolling_mean_168", "rolling_std_168",
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "month_sin", "month_cos",
    "day_sin", "day_cos",
    "dayofyear_sin", "dayofyear_cos",
    "weekofyear_sin", "weekofyear_cos",
    "quarter_sin", "quarter_cos",
]].head())



[sorted]
   prediction_unit_id  is_consumption            datetime
0                   0               0 2021-09-01 00:00:00
1                   0               0 2021-09-01 01:00:00
2                   0               0 2021-09-01 02:00:00
3                   0               0 2021-09-01 03:00:00
4                   0               0 2021-09-01 04:00:00

[lag features created]
     target       lag_1  lag_24
0     0.713         NaN     NaN
1     1.132    0.713000     NaN
2     0.490    1.132000     NaN
3     0.496    0.490000     NaN
4     0.149    0.496000     NaN
5     0.298    0.149000     NaN
6     1.271    0.298000     NaN
7    22.122    1.271000     NaN
8    64.257   22.122000     NaN
9   170.312   64.257004     NaN
10  286.533  170.311996     NaN
11  391.054  286.532990     NaN
12  480.938  391.053986     NaN
13  422.591  480.937988     NaN
14  303.120  422.591003     NaN
15  208.335  303.119995     NaN
16  225.405  208.335007     NaN
17  136.324  225.404999     NaN
18   61.90

<a id="features"></a>
## 4️ Feature Engineering

공통 Feature를 먼저 생성하고, 이후 모델별 확장 Feature를 관리하기 쉽게 정리합니다.

### 4-1. 모델별 Feature 세트 정의
- `baseline_features`는 베이스라인 공통 Feature
- `engineered_features`는 추가로 생성한 파생 Feature
  - lag: `lag_2`, `lag_3`, `lag_6`, `lag_12`, `lag_48`, `lag_72`, `lag_96`, `lag_120`, `lag_168`
  - rolling: `rolling_mean/std_3`, `rolling_mean/std_6`, `rolling_mean/std_12`, `rolling_mean/std_24`, `rolling_mean/std_48`, `rolling_mean/std_168`
  - 시계성 인코딩: `hour_sin`, `hour_cos`, `month_sin`, `month_cos`, `dow_sin`, `dow_cos`
- 전처리 파일은 이 Feature들을 모두 생성하므로, 모든 모델이 같은 feature-engineered 파일을 쓴다.

In [8]:
# =========================================================
# 10. model feature sets
# =========================================================
baseline_features = [
    "county",
    "is_business",
    "product_type",
    "hour",
    "weekday",
    "month",
    "day",
    "dayofyear",
    "weekofyear",
    "temperature",
    "dewpoint",
    "rain",
    "snowfall",
    "surface_pressure",
    "cloudcover_total",
    "windspeed_10m",
    "shortwave_radiation",
    "euros_per_mwh",
    "lowest_price_per_mwh",
    "highest_price_per_mwh",
    "temp_county",
    "solar_county",
    "direct_solar",
    "cloud_county",
    "wind_county",
    "lag_1",
    "lag_24",
]

engineered_features = [
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_quarter_start",
    "is_quarter_end",
    "days_in_month",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "lag_48",
    "lag_72",
    "lag_96",
    "lag_120",
    "lag_168",
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_mean_6",
    "rolling_std_6",
    "rolling_mean_12",
    "rolling_std_12",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_48",
    "rolling_std_48",
    "rolling_mean_168",
    "rolling_std_168",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos",
    "dayofyear_sin",
    "dayofyear_cos",
    "weekofyear_sin",
    "weekofyear_cos",
    "quarter_sin",
    "quarter_cos",
]

model_feature_sets = {
    "baseline": baseline_features.copy(),
    "xgboost": baseline_features + engineered_features,
    "lgbm": baseline_features + engineered_features,
    "gru": baseline_features + engineered_features,
    "ridge": baseline_features + engineered_features,
}

print("baseline_features count:", len(baseline_features))
print("engineered_features count:", len(engineered_features))
print("model_feature_sets:", {k: len(v) for k, v in model_feature_sets.items()})


baseline_features count: 27
engineered_features count: 42
model_feature_sets: {'baseline': 27, 'xgboost': 69, 'lgbm': 69, 'gru': 69, 'ridge': 69}


<a id="missing"></a>
## 5️ 결측치 처리

보간, 평균 등 다양한 방식으로 결측치를 처리합니다.

In [9]:
# =========================================================
# 11. 결측치 처리(평균/보간)
#   날씨 / 가격 --> 선형 보간
#   lag       --> 보간 + 그룹 평균
#   고객 정보   --> 그룹 평균
#   나머지      --> 전체 평균
# =========================================================

na_before = train.isna().sum().sort_values(ascending=False).head(20)
print("\n[결측치 상위 20개 - 처리 전]")
print(na_before)

# -------------------------------
# 1) 시간순 정렬
# -------------------------------
train = train.sort_values(["prediction_unit_id", "is_consumption", "datetime"])

# -------------------------------
# 2) 컬럼 그룹 나누기
# -------------------------------
interp_cols = [
    "temperature",
    "dewpoint",
    "rain",
    "snowfall",
    "surface_pressure",
    "cloudcover_total",
    "windspeed_10m",
    "shortwave_radiation",
    "euros_per_mwh",
    "lowest_price_per_mwh",
    "highest_price_per_mwh",
]

lag_cols = ["lag_1", "lag_24"]

mean_cols = ["eic_count", "installed_capacity"]


# -------------------------------
# 3) 보간 (시계열 데이터)
# -------------------------------
for col in interp_cols:
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )

# -------------------------------
# 4) lag 처리 (보간 + 평균)
# -------------------------------
for col in lag_cols:
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )
    train[col] = (
        train.groupby(["prediction_unit_id", "is_consumption"])[col]
             .transform(lambda s: s.fillna(s.mean()))
    )

# -------------------------------
# 5) 정적 변수 평균 처리
# -------------------------------
for col in mean_cols:
    train[col] = (
        train.groupby(["county", "is_business", "product_type"])[col]
             .transform(lambda s: s.fillna(s.mean()))
    )

# -------------------------------
# 6) 마지막 남은 결측 처리 (전체 평균)
# -------------------------------
num_cols = train.select_dtypes(include="number").columns

train[num_cols] = train[num_cols].fillna(train[num_cols].mean())

# -------------------------------
# 결과 확인
# -------------------------------
na_after = train.isna().sum().sum()
print("\n[결측치 처리 완료]")
print("남은 결측치 개수:", na_after)


[결측치 상위 20개 - 처리 전]
cloud_county              1027430
direct_solar              1027430
temp_county               1027430
wind_county               1027430
solar_county              1027430
lag_168                     23710
lag_120                     17086
lag_96                      13774
lag_72                      10464
lag_48                       7152
eic_count                    6240
installed_capacity           6240
direct_solar_radiation       4810
windspeed_10m                4810
shortwave_radiation          4810
dewpoint                     4810
temperature                  4810
snowfall                     4810
surface_pressure             4810
rain                         4810
dtype: int64

[결측치 처리 완료]
남은 결측치 개수: 0


<a id="split"></a>
## 6️ 시계열 Split

시간순 기준으로 train/valid set을 분할합니다.

In [10]:
# =========================================================
# 12. 시계열 기준 split
# =========================================================
split_date = train["datetime"].quantile(0.8)

train_processed = train[train["datetime"] < split_date].copy()
valid_processed = train[train["datetime"] >= split_date].copy()

print("\n[시계열 split 완료]")
print("split_date:", split_date)
print("train_processed:", train_processed.shape)
print("valid_processed:", valid_processed.shape)


[시계열 split 완료]
split_date: 2023-01-25 01:00:00
train_processed: (1614612, 79)
valid_processed: (403740, 79)


<a id="save"></a>
## 7️ CSV 저장

전처리된 데이터를 CSV 파일로 저장합니다.

In [11]:
# =========================================================
# 13. CSV 저장
# =========================================================
full_path = OUTPUT_DIR / "full_processed.csv"
train_path = OUTPUT_DIR / "train_processed.csv"
valid_path = OUTPUT_DIR / "valid_processed.csv"

train.to_csv(full_path, index=False)
train_processed.to_csv(train_path, index=False)
valid_processed.to_csv(valid_path, index=False)

print("\n[저장 완료]")
print("full:", full_path)
print("train:", train_path)
print("valid:", valid_path)


[저장 완료]
full: d:\kaggle\data\processed_data\full_processed.csv
train: d:\kaggle\data\processed_data\train_processed.csv
valid: d:\kaggle\data\processed_data\valid_processed.csv


In [12]:
df = pd.read_csv(DATA_DIR / "processed_data" / "full_processed.csv")

print(df.columns.tolist())
df.head()

['row_id', 'prediction_unit_id', 'county', 'is_business', 'product_type', 'is_consumption', 'target', 'datetime', 'date', 'hour', 'weekday', 'month', 'day', 'dayofyear', 'weekofyear', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end', 'days_in_month', 'data_block_id', 'eic_count', 'installed_capacity', 'temperature', 'dewpoint', 'rain', 'snowfall', 'surface_pressure', 'cloudcover_total', 'windspeed_10m', 'shortwave_radiation', 'direct_solar_radiation', 'temp_county', 'solar_county', 'direct_solar', 'cloud_county', 'wind_county', 'euros_per_mwh', 'lowest_price_per_mwh', 'highest_price_per_mwh', 'lag_1', 'lag_24', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_48', 'lag_72', 'lag_96', 'lag_120', 'lag_168', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_48', 'rolling_std_48', 'rolling_mean_168', 'rolling_std_168', 'hour_sin', 'hour_cos', '

,row_id,prediction_unit_id,county,is_business,product_type,is_consumption,target,datetime,date,hour,...,month_sin,month_cos,day_sin,day_cos,dayofyear_sin,dayofyear_cos,weekofyear_sin,weekofyear_cos,quarter_sin,quarter_cos
0,0,0,0,0,1,0,0.713,2021-09-01 00:00:00,2021-09-01,0,...,-0.866025,-0.5,0.0,1.0,-0.861693,-0.50743,-0.822984,-0.568065,1.224647e-16,-1.0
1,122,0,0,0,1,0,1.132,2021-09-01 01:00:00,2021-09-01,1,...,-0.866025,-0.5,0.0,1.0,-0.861693,-0.50743,-0.822984,-0.568065,1.224647e-16,-1.0
2,244,0,0,0,1,0,0.490,2021-09-01 02:00:00,2021-09-01,2,...,-0.866025,-0.5,0.0,1.0,-0.861693,-0.50743,-0.822984,-0.568065,1.224647e-16,-1.0
3,366,0,0,0,1,0,0.496,2021-09-01 03:00:00,2021-09-01,3,...,-0.866025,-0.5,0.0,1.0,-0.861693,-0.50743,-0.822984,-0.568065,1.224647e-16,-1.0
4,488,0,0,0,1,0,0.149,2021-09-01 04:00:00,2021-09-01,4,...,-0.866025,-0.5,0.0,1.0,-0.861693,-0.50743,-0.822984,-0.568065,1.224647e-16,-1.0
